# 2. Learn a classifier and evaluate it honestly

**Prerequisite:** Lesson 1 and basic array indexing. **You will learn:** scores, softmax, loss, gradients, training versus inference, and per-class evaluation. Allow 30–45 minutes. The small training task below is intentionally easy and synthetic; its score is not a portfolio result.

**Execution:** Run from the repository or this folder, in order, using the installed project environment. The default lesson needs no downloads, credentials, GPU, or existing model checkpoint. Toy examples are teaching demonstrations, not the real-data study.

**Learning loop:** Read the theory, predict the output, run the example, attempt the exercise, then open the worked answer.


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/results.json').is_file())
print('Repository:', ROOT.name)


Repository: Vision


## 1. A model maps inputs to scores
A classifier computes one score, or logit, for each class. Softmax converts scores into nonnegative values summing to one. The largest score determines the predicted label. A neural network learns its internal weights from examples rather than from hand-written classification rules.

We start with two artificial classes and two input features so that the learning mechanism is visible. The real application has six classes and temporal sensor inputs. Cross-entropy penalizes assigning low probability to the correct label; optimization changes weights to reduce that loss.


In [2]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(7)
x = torch.cat([torch.randn(40, 2) * .2 - 1, torch.randn(40, 2) * .2 + 1])
y = torch.cat([torch.zeros(40), torch.ones(40)]).long()
model = nn.Linear(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=.1)
loss_fn = nn.CrossEntropyLoss()
losses = []
for step in range(60):
    optimizer.zero_grad()  # Discard gradients from the previous update.
    loss = loss_fn(model(x), y)
    loss.backward()       # Differentiate loss with respect to the weights.
    optimizer.step()      # Move weights in a direction that reduces loss.
    losses.append(loss.item())
print('First/last training loss:', round(losses[0], 4), round(losses[-1], 4))
assert losses[-1] < losses[0]


First/last training loss: 1.3551 0.0481


## 2. Inference does not update weights
`eval()` selects evaluation behavior for layers such as dropout and BatchNorm. `inference_mode()` avoids gradient tracking. They serve different purposes and are usually used together for deployment.

Generate a new synthetic batch for a demonstration of held-out inference. It comes from the same simple generator, so success here says little about real sensor recordings.


In [3]:
model.eval()
held_x = torch.cat([torch.randn(10, 2) * .2 - 1, torch.randn(10, 2) * .2 + 1])
held_y = torch.cat([torch.zeros(10), torch.ones(10)]).long()
with torch.inference_mode():
    probabilities = model(held_x).softmax(1)
    predictions = probabilities.argmax(1)
print('First probabilities:', probabilities[:3].numpy().round(3))
print('Toy held-out accuracy:', (predictions == held_y).float().mean().item())
assert torch.allclose(probabilities.sum(1), torch.ones(20))


First probabilities: [[0.968 0.032]
 [0.961 0.039]
 [0.98  0.02 ]]
Toy held-out accuracy: 1.0


## 3. Accuracy can hide a neglected class
Accuracy is correct predictions divided by all predictions. Precision asks how many predictions of a class were correct; recall asks how many true members were found. F1 is their harmonic mean, `2PR / (P + R)`, with zero used here when undefined. Macro F1 averages class F1 values equally.

For nine examples of class 0 and one of class 1, a classifier that always predicts 0 has high accuracy but entirely misses class 1. Predict the accuracy before running.


In [4]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
truth = [0] * 9 + [1]
predicted = [0] * 10
print('Accuracy:', accuracy_score(truth, predicted))
print('Macro F1:', round(f1_score(truth, predicted, average='macro', zero_division=0), 3))
print('Confusion matrix (true rows, predicted columns):\n', confusion_matrix(truth, predicted))


Accuracy: 0.9
Macro F1: 0.474
Confusion matrix (true rows, predicted columns):
 [[9 0]
 [1 0]]


## 4. Connect the toy classifier to SensorNet
A convolution slides learned filters over time to find local patterns. Pooling summarizes them; linear layers map the summary to six activity scores. The width controls the number of learned feature channels. A larger network may model more patterns but requires more storage and computation.

The following repository model is randomly initialized. Checking its output shape verifies the interface, not prediction quality.


In [5]:
from edge_agent_lab.models import SensorNet
sensor = SensorNet(width=4).eval()
with torch.inference_mode():
    scores = sensor(torch.zeros(3, 9, 128))
print('Activity scores shape:', tuple(scores.shape))
print('Trainable parameters:', sum(p.numel() for p in sensor.parameters()))
assert scores.shape == (3, 6)


Activity scores shape: (3, 6)
Trainable parameters: 1870


## Exercise: separate learning from selection
Try 5 rather than 60 updates in the toy training loop. Does the loss decrease as much? Why should you not repeatedly change the training duration after inspecting the final test score?

<details><summary>Worked answer</summary>

Fewer updates generally leave a higher loss on this example. Choose training duration using validation evidence. Repeated decisions based on the test set make it part of model selection and weaken the independence of the final report. A fixed seed helps repeat a run but does not measure variability across seeds.
</details>

**Application bridge:** Read `training.py` for real fitting and checkpoint selection and `metrics.py` for six-class macro F1. Continue to [optimization](03_optimization.ipynb). The [Edge Sense notebook](../01_edge_sense.ipynb) contains real subject-held-out results and a deliberately unsuccessful transfer experiment.
